# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haiqa037/Repository_ML_internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Answer: Lane 2: Refresh / Content Opportunity Scoring.
Reason:
It’s the sole lane where the starter pipeline already demonstrates, with actual figures, that a model outperforms a baseline — providing a tangible, pre-established foundation to develop upon.




In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Starter dataset rows: {len(df):,}")
print(f"Unique clients: {df['client_id'].nunique()}")

Starter dataset rows: 30,000
Unique clients: 32


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

This study enhances the choice of content pages that a reviewer
should be examined initially, considering the restricted time. The page serves as the unit of analysis.
(content_id) throughout its previous 90-day period. The step a reviewer undertakes is
selecting a page from my ranked queue and implementing a recommended solution — reload,
enlarge, safeguard, or trim. The individual executing this is a content
strategist/reviewer with a set weekly evaluation limit (for instance, ~50
pages weekly). The expense of an incorrect decision impacts both sides: if I classify a reliable or
page that is already getting better is too elevated, causing the reviewer to spend valuable time on it while
a truly declining page sits unexamined; if I categorize a genuine decliner too
diminished, it continues to lose visibility unnoticed until the decline accumulates. Each
mistakes require valuable reviewer time and actual traffic, which is the reason ranking
precision@K holds greater significance here than mere accuracy.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df_clean = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset="content_id")
n_candidates = len(df_clean)
review_capacity = 50  # example weekly capacity
print(f"Candidate pages after filtering: {n_candidates:,}")
print(f"If a reviewer can only check {review_capacity}/week, that's "
      f"{n_candidates / review_capacity:.0f} weeks to review them all once — ranking matters.")

Candidate pages after filtering: 30,000
If a reviewer can only check 50/week, that's 600 weeks to review them all once — ranking matters.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [2]:
import os
import requests

# Define the directory and file path
data_dir = 'data/raw'
file_name = 'content_refresh_anonymized.csv'
file_path = os.path.join(data_dir, file_name)

# Create the directory if it doesn't exist
os.makedirs(data_dir, exist_ok=True)

# URL of the raw CSV file on GitHub
github_url = f'https://raw.githubusercontent.com/haiqa037/Repository_ML_internship/main/{data_dir}/{file_name}'

# Download the file
response = requests.get(github_url)
response.raise_for_status() # Raise an exception for HTTP errors

with open(file_path, 'wb') as f:
    f.write(response.content)

print(f"Downloaded '{file_name}' to '{file_path}'")

Downloaded 'content_refresh_anonymized.csv' to 'data/raw/content_refresh_anonymized.csv'


In [3]:
import pandas as pd
import json
import os
import requests

# Define the directory and file path (redundant but ensures self-containment if this cell runs independently)
data_dir = 'data/raw'
file_name = 'content_refresh_anonymized.csv'
file_path = os.path.join(data_dir, file_name)

# Check if file exists, if not, download it
if not os.path.exists(file_path):
    print(f"File '{file_path}' not found, attempting to re-download...")
    os.makedirs(data_dir, exist_ok=True)
    github_url = f'https://raw.githubusercontent.com/haiqa037/Repository_ML_internship/main/{data_dir}/{file_name}'
    response = requests.get(github_url)
    response.raise_for_status() # Raise an exception for HTTP errors
    with open(file_path, 'wb') as f:
        f.write(response.content)
    print(f"Re-downloaded '{file_name}' to '{file_path}'")

# Real numbers from the starter dataset
df = pd.read_csv(file_path)
df_clean = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset="content_id")

print(f"Rows after filtering: {len(df_clean):,}")
print(f"Share currently declining (trend_direction == 'down'): {(df_clean['trend_direction'] == 'down').mean():.1%}")
print(f"Median 90d impressions among these pages: {df_clean['impressions_90d'].median():,.0f}")

# Real numbers from the already-run starter pipeline (verified output)
# This part assumes 'outputs/model_results.json' exists. If it doesn't, this will cause a new FileNotFoundError.
# For now, keeping as is to address the primary CSV file error.
try:
    with open("outputs/model_results.json") as f:
        results = json.load(f)
    print("\nModel comparison (Precision@50):")
    for method, metrics in results.items():
        print(f"  {method}: {metrics.get('precision_at_50')}")
except FileNotFoundError:
    print("\nWarning: 'outputs/model_results.json' not found. Skipping model comparison.")

Rows after filtering: 30,000
Share currently declining (trend_direction == 'down'): 54.2%
Median 90d impressions among these pages: 731



## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

My job can identify which detected signals relate to pages that
might eventually appear as diminishing, sparse, or receiving fewer clicks, and it can position pages by
that risk so minimal evaluation time is allocated to areas where it can be most beneficial.
This is noted, directional, decision-aiding data — not evidence. I am
work cannot assure that any particular page will definitely recover upon being refreshed,
that I’ve decoded a Google ranking criterion, or that a correlation I
discover is causal. Demonstrating that a refresh leads to recovery would necessitate an
real experiment including a control group, which this dataset by itself cannot
supply. I also will not consider the starter's trend_direction == "down" label as
persistent factual basis — it serves as a present-time representative, and a more robust variant
of this project ought to forecast a truly future timeframe instead.


In [12]:
from pprint import pprint

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pprint(df_clean["trend_direction"].value_counts())
print("\nNote: trend_direction is computed from the CURRENT 90-day window, "
      "not a future window — so it's a proxy label, not a verified outcome.")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Note: trend_direction is computed from the CURRENT 90-day window, not a future window — so it's a proxy label, not a verified outcome.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.